# EXP-014 원인 분석: RRF k 20이 무엇을 올리고 내렸나

기준(EXP-011, k 60) 대비 정답 상위 5위에 들어온 질의와 빠진 질의를 결합 전 목록별 순위로 나눠 본다. 가설은 오차 분석(24_01)의 결합 손실 81개(한 목록은 정답을 상위 5위에 둠)가 살아난다는 것이고, 위험은 여러 목록의 합의로 들어오던 정답을 잃는 것이다. 판정과 관계없이 쓴다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rag.conditions import build_extractor
from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
TOP = 5

## 질의별 이동

In [3]:
cfg = load_cfg(['+exp=exp011', 'index.scope=full'])
gold = load_gold(cfg, 'dev-full')
rb = gold_ranks(read_run(RUNS / f'{PREFIX}-dev-full.csv'), gold, 10)
rc = gold_ranks(read_run(RUNS / f'{PREFIX}-k20-dev-full.csv'), gold, 10)
res = {}
for k in (1, 5):
    hb, hc = rb <= k, rc <= k
    res[k] = {'gain': (~hb & hc), 'loss': (hb & ~hc)}
    print(f'top{k}: 진입 {int(res[k]["gain"].sum())}, 이탈 {int(res[k]["loss"].sum())}')
moved = rb.fillna(99) != rc.fillna(99)
print('정답 순위가 바뀐 질의', int(moved.sum()))

top1: 진입 3, 이탈 2
top5: 진입 28, 이탈 5
정답 순위가 바뀐 질의 95


## 결합 전 목록별 정답 순위

목록(dense, BM25, 조건)은 k와 무관하므로 기준 설정으로 한 번 계산한다. 정답 순위가 바뀐 질의와 기준 실패 질의만 본다.

In [4]:
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = dict(zip(qids, read_texts(text_path, qids), strict=True))
q_all = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
q_emb = dict(zip(qids, q_all, strict=True))
ex = build_extractor(cfg)
miss_b = ~(rb <= TOP)
sub = sorted(set(moved[moved].index) | set(miss_b[miss_b].index))
tok = dict(zip(sub, ret.query_tokens([texts[q] for q in sub]), strict=True))
rows = []
for q in sub:
    named, _, _ = ret.lists(ret.chunk_emb @ q_emb[q], tok[q], ex.extract(texts[q]))
    g = gold.loc[q, 'doc_id']
    pos = {n: (lst.index(g) + 1 if g in lst else np.nan) for n, lst in named.items()}
    rows.append({'qid': q, **{f'{n}_rank': pos.get(n, np.nan) for n in ['dense', 'bm25']},
                 'has_cond': 'conditions' in named})
lr = pd.DataFrame(rows).set_index('qid')
lr['base'], lr['k20'] = rb[sub], rc[sub]
best = lr[['dense_rank', 'bm25_rank']].min(axis=1)
worst = lr[['dense_rank', 'bm25_rank']].max(axis=1)
lr['type'] = np.where(best <= TOP, np.where(worst <= TOP, '두 목록 top5', '한 목록만 top5'),
                      '어느 목록도 top5 아님')
print(len(lr), 'queries')

225 queries


In [5]:
for name in ['gain', 'loss']:
    s = lr.loc[[q for q in sub if res[TOP][name][q]]]
    print(f'top{TOP} {name} {len(s)}')
    print(s['type'].value_counts().to_string())
    print(s[['dense_rank', 'bm25_rank']].median().round(1).to_dict(), '\n')

top5 gain 28
type
한 목록만 top5    18
두 목록 top5     10
{'dense_rank': 32.5, 'bm25_rank': 1.0} 

top5 loss 5
type
어느 목록도 top5 아님    5
{'dense_rank': 30.0, 'bm25_rank': 16.0} 



### 오차 분석의 결합 손실 81개

기준 실패 중 한 목록은 정답을 상위 5위에 둔 질의(24_01과 같은 정의, 조건 목록 포함 대신 dense·BM25만 본다)가 몇 개 살아났는지.

In [6]:
one_list = lr.index[(lr[['dense_rank', 'bm25_rank']].min(axis=1) <= TOP)]
fusion_loss = [q for q in one_list if miss_b[q]]
print('기준 실패 중 dense·BM25 한 목록은 top5', len(fusion_loss))
print('그중 k 20에서 top5', int((rc[fusion_loss] <= TOP).sum()))

기준 실패 중 dense·BM25 한 목록은 top5 81
그중 k 20에서 top5 28


### 잃은 질의는 합의로 들어오던 정답인가

잃은 질의의 정답이 기준에서 두 목록 모두 중간 순위(어느 목록도 1~2위가 아님)였는지 본다.

In [7]:
lost = lr.loc[[q for q in sub if res[TOP]['loss'][q]]]
both = lost[['dense_rank', 'bm25_rank']]
mid = (both.min(axis=1) > 2) & both.notna().all(axis=1)
print('잃은 질의', len(lost), ' 두 목록에 모두 있고 어느 쪽도 1~2위가 아님', int(mid.sum()))
print(lost[['dense_rank', 'bm25_rank', 'base', 'k20']].to_string())

잃은 질의 5  두 목록에 모두 있고 어느 쪽도 1~2위가 아님 5
             dense_rank  bm25_rank  base  k20
qid                                          
043711-0001        61.0       16.0   1.0  9.0
046680-0017        18.0        7.0   4.0  6.0
047529-0009        13.0       16.0   5.0  6.0
050448-0007        30.0       10.0   5.0  6.0
052371-0003        42.0       30.0   5.0  NaN


## 정리

- 상위 5위 진입 28, 이탈 5(순증 23, Recall@5 +0.76%p). 1위는 진입 3·이탈 2로 거의 그대로다
- 결합 손실 81개(기준 실패 중 dense·BM25 한 목록은 정답을 top5에 둠) 중 28개(35%)가 살아났다. 살아난 28개 중 18개는 한 목록만 top5였고 BM25 쪽이 중심이다(중앙 BM25 1위, dense 32.5위). 나머지 10개는 두 목록 모두 top5였는데도 k 60에서 밀려 있었다
- 잃은 5개는 모두 어느 목록에서도 top5가 아니었고 두 목록 모두에 중간 순위로 있었다(BM25 7~30위, dense 13~61위). 합의로 들어오던 정답을 잃는다는 위험이 실제로 일어났지만 5개에 그쳤다
- 결합 손실 53개는 k 20으로도 살아나지 않았다. 남은 몫을 겨냥하려면 순위만 쓰는 RRF 밖의 방식(점수 기반 결합, 재정렬)이 필요하다